In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import pickle
import tensorflow as tf
import matplotlib.pyplot as plt
from itertools import cycle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, auc
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from keras.layers import Input, Conv1D, MaxPooling1D, UpSampling1D, concatenate, BatchNormalization, Activation, add
from keras.layers import Conv2D, MaxPooling2D, Reshape, Flatten, Dense
from keras.models import Model, model_from_json
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint
sns.set_theme(style="whitegrid")
import ast

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
train_pkl_file = r'/content/drive/MyDrive/Data/New_VitalDB_2peaks_Onset.pkl'
df_train = pd.read_pickle(train_pkl_file)
print (df_train)

                                               Cropped_1s  Peak_Index  \
0       [-10.875439998578193, -11.142804467733681, -11...         563   
1       [-12.414277921858734, -12.301388696403217, -11...         287   
2       [-9.550334092908498, -9.535268964451095, -9.52...         602   
3       [-6.97291877026109, -7.246176236715118, -7.484...         384   
4       [-9.482280671654904, -9.090561717384709, -8.78...         156   
...                                                   ...         ...   
166523  [-10.52681159055055, -10.684670971560417, -10....         809   
166524  [-10.358293662629745, -10.68214189005744, -10....         544   
166525  [-10.115408395251016, -10.433467331586131, -10...         422   
166526  [-13.493630264611479, -13.514172277356975, -13...         108   
166527  [-12.033259830222507, -12.499298357365303, -13...         586   

       Onset_Index  Middle_Index  Case     dt  Segment_Number  Result  \
0       (540, 615)           578     1   3060     

In [ ]:
df_train['VPG'] = df_train['Cropped_1s'].apply(lambda x: np.diff(x))
df_train['APG'] = df_train['VPG'].apply(lambda x: np.diff(x))
df_train['VPG_Signal_Normalized'] = df_train['Cropped_1s_Normalized'].apply(lambda x: np.diff(x))
df_train['APG_Signal_Normalized'] = df_train['VPG_Signal_Normalized'].apply(lambda x: np.diff(x))
print (df_train)
df_train.to_pickle(r'/content/drive/MyDrive/Data/Green_Inverted_Onset_v2.pkl')

                                            Cropped_1s  Peak_Index  \
0    [-3.002764414574935, -8.000947137837105, -11.7...         146   
1    [-43.9325485614622, -44.47639870997052, -43.44...         657   
2    [32.68479051832884, 24.69729298630475, 18.1779...         868   
3    [-150.17061004726563, -158.69337197959325, -16...         129   
4    [69.86364977782648, 83.62747769944863, 88.9382...         244   
..                                                 ...         ...   
256  [-18.642593122768208, -18.885189456384364, -20...         510   
257  [4.464590686648956, -2.4617851618173057, -9.34...         637   
258  [-41.10158132010425, -44.29591232620037, -48.5...         341   
259  [-27.939839696605656, -28.685685523912014, -31...         532   
260  [-31.265260102532213, -38.00939200503329, -46....         793   

    Onset_Index  Middle_Index  Segment_Number  Result  Age File_name  \
0    (133, 201)           167               1     311   42   05a.csv   
1    (614, 712)

In [ ]:
def extract_full_pulse_segments(PPG_filter, peak_index, onset_index):
    """
    Extract 1-second PPG segments containing full pulses.

    Each segment captures a complete cardiac cycle: onset(j) < peak(i) < onset(j+1)
    The segment is centered on the middle point between consecutive onsets.

    Parameters:
    -----------
    PPG_filter : array-like
        Filtered PPG signal
    peak_index : array-like
        Indices of detected peaks
    onset_index : array-like
        Indices of detected onsets
    halfwin : int, default=50
        Half window size (total segment length = 2*halfwin = 100 samples)


    Returns:
    --------
    all_1s_signal : list
        List of extracted 1-second PPG segments
    midde_indices_in_segments : list
        Midde indices to each segment
    onset_pairs : list
        Pairs of onset indices used for each segment

    """

    # Convert inputs to numpy arrays
    PPG_filter = np.array(PPG_filter)
    peak_index = np.array(peak_index)
    onset_index = np.array(onset_index)
    halfwin = 50  # Half window size for segment extraction

    # Initialize output variables
    all_1s_signal = []
    middle_indices_in_segments = []
    onset_pairs = []
    peak_indexes = []
    index_new = 0

    # Sort indices to ensure proper ordering
    peak_index = np.sort(peak_index)
    onset_index = np.sort(onset_index)

    # Process each peak to find corresponding onset pair
    for i in range(len(peak_index)):
        # Get current peak index
        current_peak = peak_index[i]

        # Find the onset pair that contains this peak
        for j in range(len(onset_index) - 1):
            onset_j = onset_index[j]
            onset_j_plus_1 = onset_index[j + 1]

            # Check if peak is between consecutive onsets
            if onset_j < current_peak < onset_j_plus_1:
                # Calculate middle point between onsets (center of cardiac cycle)
                middle_index = round((onset_j + onset_j_plus_1) / 2)

                # Calculate segment boundaries centered on middle point
                start_idx = middle_index - halfwin
                end_idx = middle_index + halfwin

                # Check bounds and segment length
                if (start_idx >= 0 and
                    end_idx < len(PPG_filter) and end_idx - start_idx == 2 * halfwin):

                    # Extract segment
                    segment = PPG_filter[start_idx:end_idx]  # +1 for inclusive end

                    # Store results
                    all_1s_signal.append(segment)
                    middle_indices_in_segments.append(middle_index)
                    peak_indexes.append(current_peak)
                    onset_pairs.append((onset_j, onset_j_plus_1))
                    index_new += 1

                    break  # Found the onset pair for this peak, move to next peak

    return all_1s_signal, middle_indices_in_segments, peak_indexes, onset_pairs, index_new

In [ ]:
new_rows = []
for _, row in df_expanded.iterrows():
    signal = row['PPG_Segment_Reflected']
    No = row['No.']
    Age = row['Age']
    filename = row['File_name']
    segment = row['Segment_Number']
    indices = row['Peaks_Onsets']
    Glucose = row['Glucose']
    peak_indices = indices[0]
    onset_indices = indices[1]

    all_1s_signal, middle_indices, peak_indexes, onset_pairs, _ = extract_full_pulse_segments(signal, peak_indices, onset_indices)
    new_rows.append({
                'No.': No,
                'Age': Age,
                'File_name': filename,
                'Segment_Number': segment,
                'Glucose': Glucose,
                'All_1s_Signal': all_1s_signal,
                'Middle_index' : middle_indices,
                'Peak_Index': peak_indexes,
                'Onset_Index': onset_pairs
            })

df_new = pd.DataFrame(new_rows)
print(df_new)

     No.   Age File_name  Segment_Number  Glucose  \
0      0  75.0    6a.csv               0     97.0   
1      0  75.0    6a.csv               1     97.0   
2      0  75.0    6a.csv               2     97.0   
3      0  75.0    6a.csv               3     97.0   
4      0  75.0    6a.csv               4     97.0   
..   ...   ...       ...             ...      ...   
127   25  65.0   37b.csv               1    176.0   
128   25  65.0   37b.csv               2    176.0   
129   25  65.0   37b.csv               3    176.0   
130   25  65.0   37b.csv               4    176.0   
131   25  65.0   37b.csv               5    176.0   

                                         All_1s_Signal  \
0    [[0.6743583790622717, 3.9053555405841567, 6.72...   
1    [[-80.45516314211625, -79.62357224589547, -78....   
2    [[-71.0859812454403, -68.02178816370447, -65.0...   
3    [[-80.30521603412338, -76.09588790515811, -69....   
4    [[-65.39134642763528, -62.833659327519015, -59...   
..             

In [ ]:
columns_to_keep = ['No.', 'Age', 'File_name', 'Segment_Number', 'Glucose']

rows = []

for i, row in df_new.iterrows():
    signal_list = row['All_1s_Signal']
    peak_list = row['Peak_Index']
    onset_list = row['Onset_Index']
    middle_list = row['Middle_index']
    for j, segment in enumerate(signal_list):
            rows.append({
                'Cropped_1s': segment,
                'Peak_Index': peak_list[j],
                'Onset_Index': onset_list[j],
                'Middle_Index': middle_list[j],
                **{col: row[col] for col in columns_to_keep}
            })

df_flattened = pd.DataFrame(rows)

In [ ]:
df_flattened = df_flattened[df_flattened['Cropped_1s'].notna()]
print (df_flattened)

                                             Cropped_1s  Peak_Index  \
0     [0.6743583790622717, 3.9053555405841567, 6.723...          74   
1     [-31.315241027489197, -38.33073726356154, -45....         146   
2     [-47.90389890167844, -49.77543239206435, -51.8...         218   
3     [-75.36527751516341, -73.61327585998531, -70.4...         316   
4     [-79.53596554208524, -75.16300025830014, -70.6...         417   
...                                                 ...         ...   
1451  [-195.6415355121455, -205.27195251804795, -213...         566   
1452  [-163.1050064410683, -182.18984263594444, -201...         648   
1453  [-140.57072997035885, -148.08550914711694, -15...         719   
1454  [-235.09448264986102, -250.293205289544, -264....         800   
1455  [-185.78607608972456, -201.2934329945285, -218...         882   

     Onset_Index  Middle_Index  No.   Age File_name  Segment_Number  Glucose  
0      (47, 123)            85    0  75.0    6a.csv               0 

In [ ]:
filtered_df = df_flattened.merge(
    df_train_2[['No.', 'Age', 'Segment_Number', 'Glucose', 'Peak_Index']].drop_duplicates(),
    left_on=['No.', 'Age', 'Segment_Number', 'Glucose', 'Peak_Index'],
    right_on=['No.', 'Age', 'Segment_Number', 'Glucose', 'Peak_Index']
)

In [ ]:
import os
import pandas as pd
import numpy as np
import re

# Folder path
folder_path = r"/content/drive/MyDrive/PPG_Signal_16mins"

# Store results
rows = []

# Loop through each CSV file
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        # Extract X (case) and Y (dt) using regex
        match = re.match(r"PPG_16mins_case_(\d+)_dt_(\d+)\.csv", filename)
        if match:
            case = int(match.group(1))
            dt = int(match.group(2))

            # Read the CSV file
            filepath = os.path.join(folder_path, filename)
            df = pd.read_csv(filepath)

            # Extract PPG signal and result
            ppg_signal = df['PPG_Signal'].to_numpy()
            result = df['result'].iloc[0]  # all values are the same

            # Append as a new row
            rows.append({
                "Case": case,
                "dt": dt,
                "PPG_Signal": ppg_signal,
                "result": result
            })

# Create final DataFrame
df_combined = pd.DataFrame(rows)
print (df_combined)

      Case  dt                                         PPG_Signal  result
0     5780   1  [30.8396, 30.0497, 29.6547, 29.2597, 28.8647, ...    93.0
1     5780   2  [28.4697, 28.0748, 28.0748, 27.6798, 27.6798, ...    93.0
2     5780   4  [37.9493, 37.1594, 36.7644, 36.3694, 36.3694, ...   125.0
3     5781   1  [36.7644, 36.7644, 37.1594, 37.1594, 36.7644, ...    99.0
4     5781   2  [36.7644, 36.7644, 36.7644, 37.1594, 37.1594, ...   115.0
...    ...  ..                                                ...     ...
7260   218   6  [36.36940002441406, 36.36940002441406, 36.7644...   141.0
7261   218   7  [36.76440048217773, 36.76440048217773, 36.7644...   145.0
7262   218   8  [36.76440048217773, 36.36940002441406, 36.7644...   146.0
7263   221   1  [39.52930068969727, 39.52930068969727, 39.1343...    91.0
7264   221   2  [43.08409881591797, 43.08409881591797, 43.4790...    92.0

[7265 rows x 4 columns]


In [ ]:
from scipy.signal import resample_poly
from math import gcd

final_df['IR channel_100'] = final_df['IR channel'].apply(lambda x: resample_poly(x, up = 2, down =1))
final_df['Red channel_100'] = final_df['Red channel'].apply(lambda x: resample_poly(x, up = 2, down =1))
final_df['Green channel_100'] = final_df['Green channel'].apply(lambda x: resample_poly(x, up = 2, down =1))
print (final_df)

   filename                                         IR channel  \
0   05a.csv  [43181.5, 43174.0, 43162.5, 43157.0, 43154.5, ...   
1   05b.csv  [31600.5, 31624.0, 31658.5, 31658.5, 31638.5, ...   
2   09a.csv  [35802.5, 35786.0, 35786.0, 35794.0, 35817.0, ...   
3   13a.csv  [36772.5, 36811.0, 37022.0, 37037.0, 36705.5, ...   
4   13b.csv  [41341.5, 41341.5, 41328.5, 41322.0, 41321.0, ...   
5   17a.csv  [53104.0, 53071.5, 53184.0, 53289.5, 53345.0, ...   
6   18b.csv  [52290.0, 52287.5, 52283.5, 52274.0, 52272.5, ...   
7   19a.csv  [37356.0, 37357.0, 37364.0, 37349.0, 37342.0, ...   
8   20b.csv  [49020.0, 49036.0, 49035.0, 49029.0, 49041.0, ...   
9   21b.csv  [52911.0, 52907.0, 52900.0, 52897.0, 52885.0, ...   
10  26a.csv  [48198.0, 48207.0, 48216.0, 48221.0, 48212.0, ...   
11  26b.csv  [53672.0, 53659.0, 53651.0, 53644.0, 53634.0, ...   
12  30a.csv  [44231.0, 44191.0, 44173.0, 44170.0, 44168.0, ...   
13  30b.csv  [44325.0, 44341.0, 44345.0, 44346.0, 44349.0, ...   
14  33b.cs

In [ ]:
final_df['Green_Centered'] = final_df['Green channel_100'].apply(lambda x: x - np.mean(x))
print (final_df)

   filename                                         IR channel  \
0   05a.csv  [43181.5, 43174.0, 43162.5, 43157.0, 43154.5, ...   
1   05b.csv  [31600.5, 31624.0, 31658.5, 31658.5, 31638.5, ...   
2   09a.csv  [35802.5, 35786.0, 35786.0, 35794.0, 35817.0, ...   
3   13a.csv  [36772.5, 36811.0, 37022.0, 37037.0, 36705.5, ...   
4   13b.csv  [41341.5, 41341.5, 41328.5, 41322.0, 41321.0, ...   
5   17a.csv  [53104.0, 53071.5, 53184.0, 53289.5, 53345.0, ...   
6   18b.csv  [52290.0, 52287.5, 52283.5, 52274.0, 52272.5, ...   
7   19a.csv  [37356.0, 37357.0, 37364.0, 37349.0, 37342.0, ...   
8   20b.csv  [49020.0, 49036.0, 49035.0, 49029.0, 49041.0, ...   
9   21b.csv  [52911.0, 52907.0, 52900.0, 52897.0, 52885.0, ...   
10  26a.csv  [48198.0, 48207.0, 48216.0, 48221.0, 48212.0, ...   
11  26b.csv  [53672.0, 53659.0, 53651.0, 53644.0, 53634.0, ...   
12  30a.csv  [44231.0, 44191.0, 44173.0, 44170.0, 44168.0, ...   
13  30b.csv  [44325.0, 44341.0, 44345.0, 44346.0, 44349.0, ...   
14  33b.cs

In [ ]:
import numpy as np

def remove_outliers(signal: np.ndarray):
    """
    Return None if any abs(value) > threshold_factor * mean(abs(signal)),
    otherwise return the original signal.
    """
    abs_mean = np.mean(np.abs(signal))
    threshold = 3*abs_mean
    return signal[np.abs(signal) <= threshold]

df_train = final_df
df_train['Green_Cut'] = df_train['Green_Centered'].apply(lambda x: remove_outliers(x))
print (df_train)

   filename                                         IR channel  \
0   05a.csv  [43181.5, 43174.0, 43162.5, 43157.0, 43154.5, ...   
1   05b.csv  [31600.5, 31624.0, 31658.5, 31658.5, 31638.5, ...   
2   09a.csv  [35802.5, 35786.0, 35786.0, 35794.0, 35817.0, ...   
3   13a.csv  [36772.5, 36811.0, 37022.0, 37037.0, 36705.5, ...   
4   13b.csv  [41341.5, 41341.5, 41328.5, 41322.0, 41321.0, ...   
5   17a.csv  [53104.0, 53071.5, 53184.0, 53289.5, 53345.0, ...   
6   18b.csv  [52290.0, 52287.5, 52283.5, 52274.0, 52272.5, ...   
7   19a.csv  [37356.0, 37357.0, 37364.0, 37349.0, 37342.0, ...   
8   20b.csv  [49020.0, 49036.0, 49035.0, 49029.0, 49041.0, ...   
9   21b.csv  [52911.0, 52907.0, 52900.0, 52897.0, 52885.0, ...   
10  26a.csv  [48198.0, 48207.0, 48216.0, 48221.0, 48212.0, ...   
11  26b.csv  [53672.0, 53659.0, 53651.0, 53644.0, 53634.0, ...   
12  30a.csv  [44231.0, 44191.0, 44173.0, 44170.0, 44168.0, ...   
13  30b.csv  [44325.0, 44341.0, 44345.0, 44346.0, 44349.0, ...   
14  33b.cs

In [ ]:
import numpy as np
from typing import List, Tuple
from scipy.signal import butter, filtfilt, find_peaks
def butter_bandpass_filter(
    data: np.ndarray,
    lowcut: float = 0.5,
    highcut: float = 8.0,
    fs: int = 100,
    order: int = 4
) -> np.ndarray:
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

In [ ]:
df_train['Green_Filtered'] = df_train['Green_Cut'].apply(lambda x: butter_bandpass_filter(x))
print (df_train)

   filename                                         IR channel  \
0   05a.csv  [43181.5, 43174.0, 43162.5, 43157.0, 43154.5, ...   
1   05b.csv  [31600.5, 31624.0, 31658.5, 31658.5, 31638.5, ...   
2   09a.csv  [35802.5, 35786.0, 35786.0, 35794.0, 35817.0, ...   
3   13a.csv  [36772.5, 36811.0, 37022.0, 37037.0, 36705.5, ...   
4   13b.csv  [41341.5, 41341.5, 41328.5, 41322.0, 41321.0, ...   
5   17a.csv  [53104.0, 53071.5, 53184.0, 53289.5, 53345.0, ...   
6   18b.csv  [52290.0, 52287.5, 52283.5, 52274.0, 52272.5, ...   
7   19a.csv  [37356.0, 37357.0, 37364.0, 37349.0, 37342.0, ...   
8   20b.csv  [49020.0, 49036.0, 49035.0, 49029.0, 49041.0, ...   
9   21b.csv  [52911.0, 52907.0, 52900.0, 52897.0, 52885.0, ...   
10  26a.csv  [48198.0, 48207.0, 48216.0, 48221.0, 48212.0, ...   
11  26b.csv  [53672.0, 53659.0, 53651.0, 53644.0, 53634.0, ...   
12  30a.csv  [44231.0, 44191.0, 44173.0, 44170.0, 44168.0, ...   
13  30b.csv  [44325.0, 44341.0, 44345.0, 44346.0, 44349.0, ...   
14  33b.cs

In [ ]:
import numpy as np
import pandas as pd

def split_into_10s_segments(signal: np.ndarray, fs: int = 100) -> np.ndarray:
    segment_len = fs * 10  # 10 seconds worth of samples
    n_segments = len(signal) // segment_len
    return np.reshape(signal[:n_segments * segment_len], (n_segments, segment_len))

df_train = df_merged
expanded_rows = []

for _, row in df_train.iterrows():
    signal = row['Green_Filtered']
    age = row['age']
    filename = row['filename']
    result = row['bloodGlucose']

    segments = split_into_10s_segments(signal, fs=100)

    for i, segment in enumerate(segments):
        expanded_rows.append({
            'Segment_Number': i,
            'Age': age,
            'File_name': filename,
            'Glucose': result,
            'PPG_Segment': segment
        })
df_expanded = pd.DataFrame(expanded_rows)
print(df_expanded)

     Segment_Number  Age File_name  Glucose  \
0                 0   42   05a.csv      311   
1                 1   42   05a.csv      311   
2                 2   42   05a.csv      311   
3                 3   42   05a.csv      311   
4                 4   42   05a.csv      311   
..              ...  ...       ...      ...   
106               1   72   35b.csv      313   
107               2   72   35b.csv      313   
108               3   72   35b.csv      313   
109               4   72   35b.csv      313   
110               5   72   35b.csv      313   

                                           PPG_Segment  
0    [13.356884608444958, -9.148327623185136, -28.7...  
1    [39.13716395126545, 39.99351715385333, 41.2936...  
2    [43.935461835001455, 37.216446019551505, 30.22...  
3    [14.179323261683749, -2.3363617837503288, -22....  
4    [21.520734170059207, 30.81025673404984, 39.525...  
..                                                 ...  
106  [-1.4868188628763694, -12.76408

In [ ]:
df_expanded['PPG_Segment_Reflected'] = df_expanded['PPG_Segment'].apply(lambda x: -x)
print (df_expanded)

     Segment_Number  Age File_name  Glucose  \
0                 0   42   05a.csv      311   
1                 1   42   05a.csv      311   
2                 2   42   05a.csv      311   
3                 3   42   05a.csv      311   
4                 4   42   05a.csv      311   
..              ...  ...       ...      ...   
106               1   72   35b.csv      313   
107               2   72   35b.csv      313   
108               3   72   35b.csv      313   
109               4   72   35b.csv      313   
110               5   72   35b.csv      313   

                                           PPG_Segment  \
0    [13.356884608444958, -9.148327623185136, -28.7...   
1    [39.13716395126545, 39.99351715385333, 41.2936...   
2    [43.935461835001455, 37.216446019551505, 30.22...   
3    [14.179323261683749, -2.3363617837503288, -22....   
4    [21.520734170059207, 30.81025673404984, 39.525...   
..                                                 ...   
106  [-1.4868188628763694, -1

In [ ]:
def z_score_normalize(signal: np.ndarray) -> np.ndarray:
    mean = np.mean(signal)
    std = np.std(signal)
    return (signal - mean) / std

In [ ]:
df_train['APG_Normalized_Trial'] = df_train['APG'].apply(
    lambda x: z_score_normalize(x)
)
print (df_train)

         caseid     dt  segment  Result  Index_Peak  Value_Peak  \
210           1   3060       41   154.0         563   20.597594   
2984          4  19934       23   180.0         287   24.192576   
3081          7   4092       11   103.0         602   21.551119   
3099          7   4092       14   103.0         902   23.580233   
3103          7   4092       15   103.0         384   21.035355   
...         ...    ...      ...     ...         ...         ...   
2102210    6388   3503       23    90.0         932   29.357240   
2102699    6388   6649        8   116.0         910   27.055843   
2102728    6388   6649       19   116.0          57   25.460336   
2102765    6388   6649       27   116.0         108   21.586047   
2102811    6388   6649       53   116.0         586   20.738425   

                                             PPG_Signal_1s Detected_Peaks  \
210      [-1.109369622988822, -2.5977141009420235, -3.9...       [50, 89]   
2984     [-2.1507741124524453, -2.8561634

In [ ]:
import numpy as np
from typing import List, Tuple
from scipy.signal import butter, filtfilt, find_peaks
def detect_center_peaks(
    segment: np.ndarray,
    height_thresh: float = 20,
    distance_thresh: int = 80
) -> np.ndarray:
    peaks, _ = find_peaks(segment, height=height_thresh, distance=distance_thresh)
    return peaks

def crop_1s_around_peaks(
    segment: np.ndarray,
    peaks: np.ndarray,
    window_size: int = 100
) -> List[np.ndarray]:
    half_win = window_size // 2
    cropped_segments = []
    for peak in peaks:
        start = peak - half_win
        end = peak + half_win
        if start >= 0 and end <= len(segment):
            cropped_segments.append(segment[start:end])
    return cropped_segments

def detect_peaks_with_prominence(
    ppg_segment: np.ndarray,
    height_thresh: int = 5,
    distance_thresh: int = 10,
    prominence_thresh: float = 1
) -> np.ndarray:
    peaks, _ = find_peaks(
        ppg_segment,
        height = height_thresh,
        distance=distance_thresh,
        prominence=prominence_thresh
    )
    return peaks

In [ ]:
window_size = 100
half_win = window_size // 2
df_expanded['Peak_Constant'] = df_expanded['PPG_Segment_Reflected'].apply(lambda x: detect_center_peaks(x))
print (df_expanded)

     Segment_Number  Age File_name  Glucose  \
0                 0   42   05a.csv      311   
1                 1   42   05a.csv      311   
2                 2   42   05a.csv      311   
3                 3   42   05a.csv      311   
4                 4   42   05a.csv      311   
..              ...  ...       ...      ...   
106               1   72   35b.csv      313   
107               2   72   35b.csv      313   
108               3   72   35b.csv      313   
109               4   72   35b.csv      313   
110               5   72   35b.csv      313   

                                           PPG_Segment  \
0    [13.356884608444958, -9.148327623185136, -28.7...   
1    [39.13716395126545, 39.99351715385333, 41.2936...   
2    [43.935461835001455, 37.216446019551505, 30.22...   
3    [14.179323261683749, -2.3363617837503288, -22....   
4    [21.520734170059207, 30.81025673404984, 39.525...   
..                                                 ...   
106  [-1.4868188628763694, -1

In [ ]:
cropped_rows = []
window_size = 100
half_win = window_size // 2

for _, row in df_expanded.iterrows():
    signal = row['PPG_Segment_Reflected']
    peak_indices = row['Peak_Constant']
    Segment_Number = row['Segment_Number']
    Age = row['Age']
    filename = row['File_name']
    Result = row['Glucose']

    for peak in peak_indices:
        start = peak - half_win
        end = peak + half_win

        if start >= 0 and end <= len(signal):
            cropped_rows.append({
                'Peak_Index': peak,
                'Cropped_1s': signal[start:end],
                'Segment_Number': Segment_Number,
                'Age': Age,
                'File_name': filename,
                'Result': Result
            })
df_cropped_1s = pd.DataFrame(cropped_rows)
print(df_cropped_1s)

     Peak_Index                                         Cropped_1s  \
0            70  [-28.944887850048943, -35.291063187471515, -42...   
1           275  [-17.75241283611366, -21.87310276826311, -26.8...   
2           416  [-6.895838186491882, -8.6073482059729, -12.488...   
3           555  [-25.51235523275042, -34.80713994299696, -43.2...   
4           754  [-18.325346302708837, -26.970298651595563, -32...   
..          ...                                                ...   
690         148  [20.899623220821162, 10.604718258571062, 1.200...   
691         341  [10.259230647139846, -0.8589517313291105, -8.0...   
692         532  [11.909442386703075, 2.031948502417702, -4.456...   
693         666  [2.7944966287699, 4.428829225920533, 4.2230984...   
694         793  [-12.352650810036561, -13.228571579431794, -13...   

     Segment_Number  Age File_name  Result  
0                 0   42   05a.csv     311  
1                 0   42   05a.csv     311  
2                 0   42

In [ ]:
rows = []

for idx, row in df_cropped_1s.iterrows():
     Peak_Index = row['Peak_Index']
     seg_1s = row['Cropped_1s']
     Age = row['Age']
     Segment_Number = row['Segment_Number']
     filename = row['File_name']
     Result = row['Result']

     peaks = detect_peaks_with_prominence(seg_1s)
     if len(peaks) >= 2:
                rows.append({
                    'Peak_Index': Peak_Index,
                    'Cropped_1s': seg_1s,
                    'Age': Age,
                    'File_name': filename,
                    'Segment_Number': Segment_Number,
                    'Result': Result
                })

df_2peaks = pd.DataFrame(rows)
print(df_2peaks)

     Peak_Index                                         Cropped_1s  Age  \
0           146  [-25.363644044618486, -27.18942826733997, -27....   42   
1           657  [-26.096589536231818, -29.161720632023012, -32...   42   
2           868  [-23.638601649643704, -31.786083568675572, -39...   42   
3           129  [-35.04928976458896, -40.58128174815681, -47.2...   49   
4           244  [-15.637808516751932, -23.936839127538782, -34...   49   
..          ...                                                ...  ...   
285         510  [0.5088445136331086, -2.2731624624956757, -3.9...   72   
286         637  [-15.728860610055497, -19.388931780279588, -20...   72   
287         341  [10.259230647139846, -0.8589517313291105, -8.0...   72   
288         532  [11.909442386703075, 2.031948502417702, -4.456...   72   
289         793  [-12.352650810036561, -13.228571579431794, -13...   72   

    File_name  Segment_Number  Result  
0     05a.csv               1     311  
1     05a.csv      

In [ ]:
from scipy.signal import detrend

def msptd_beat_detector(sig, fs):
    """
    MSPTD PPG beat detector.

    MSPTD_BEAT_DETECTOR detects beats in a photoplethysmogram (PPG) signal
    using the 'Multi-Scale Peak and Trough Detection' beat detector

    Parameters
    ----------
    sig : array-like
        A vector of PPG values
    fs : float
        The sampling frequency of the PPG in Hz

    Returns
    -------
    peaks : numpy.ndarray
        Indices of detected pulse peaks
    onsets : numpy.ndarray
        Indices of detected pulse troughs (i.e. onsets)

    Reference
    ---------
    S. M. Bishop and A. Ercole, 'Multi-scale peak and trough detection optimised
    for periodic and quasi-periodic neuroscience data,' in Intracranial Pressure
    and Neuromonitoring XVI. Acta Neurochirurgica Supplement, T. Heldt, Ed.
    Springer, 2018, vol. 126, pp. 189-195.
    <https://doi.org/10.1007/978-3-319-65798-1_39>

    Author
    ------
    Peter H. Charlton (Original MATLAB)
    Converted to Python
    """

    # Convert to numpy array
    # Already a numpy array


    # Window signal
    # Split into overlapping 6 s windows
    win_len = 6  # in secs
    overlap = 0.2  # proportion of overlap between consecutive windows
    no_samps_in_win = int(win_len * fs)

    # The windows include the entire signal duration
    win_offset = round(no_samps_in_win * (1 - overlap))
    win_starts = np.arange(0, len(sig) - no_samps_in_win, win_offset)
    win_ends = win_starts + no_samps_in_win

    if win_ends[-1] < len(sig):
        win_starts = np.append(win_starts, len(sig) - no_samps_in_win)
        win_ends = np.append(win_ends, len(sig))

    # Detect peaks and onsets in each window
    peaks = []
    onsets = []

    for win_no in range(len(win_starts)):
        # Extract this window's data
        win_sig = sig[win_starts[win_no]:win_ends[win_no]]

        rel_sig = win_sig.copy()

        # Detect peaks and onsets
        p, t = detect_peaks_and_onsets_using_msptd(win_sig)

        # Correct peak indices by finding highest point within tolerance either side of detected peaks
        tol = int(np.ceil(fs * 0.05))
        for pk_no in range(len(p)):
            curr_peak = p[pk_no]
            tol_start = curr_peak - tol
            tol_end = curr_peak + tol
            temp = np.argmax(rel_sig[tol_start:tol_end])
            p[pk_no] = curr_peak - tol + temp

        # Correct onset indices by finding lowest point within tolerance either side of detected onsets
        for onset_no in range(len(t)):
            curr_onset = t[onset_no]
            tol_start = curr_onset - tol
            tol_end = curr_onset + tol
            temp = np.argmin(rel_sig[tol_start:tol_end])
            t[onset_no] = curr_onset - tol + temp

        # Store peaks and onsets
        win_peaks = p + win_starts[win_no]
        peaks.extend(win_peaks)
        win_onsets = t + win_starts[win_no]
        onsets.extend(win_onsets)

    # Tidy up detected peaks and onsets (by ordering them and only retaining unique ones)
    peaks = np.unique(np.array(peaks))
    onsets = np.unique(np.array(onsets))

    # Correct peak and onset indices
    # Ensure it's a column vector and only keep unique values
    peaks = np.unique(peaks.flatten())
    onsets = np.unique(onsets.flatten())

    return peaks, onsets


def detect_peaks_and_onsets_using_msptd(x):
    """
    Multi-Scale Peak and Trough Detection (MSPTD) algorithm.

    This function detects peaks and onsets (troughs) in a signal using the
    Multi-Scale Peak and Trough Detection algorithm.

    Parameters
    ----------
    x : numpy.ndarray
        Input signal

    Returns
    -------
    p : numpy.ndarray
        Detected peak indices
    t : numpy.ndarray
        Detected onset (trough) indices
    """
    from scipy.signal import detrend

    N = len(x)  # length of signal
    L = int(np.ceil(N/2)) - 1  # max window length

    # Step 1: calculate local maxima and local minima scalograms
    # - detrend
    x = detrend(x)  # this removes the best-fit straight line

    # - initialise LMS matrices
    m_max = np.zeros((L, N), dtype=bool)
    m_min = np.zeros((L, N), dtype=bool)

    # - populate LMS matrices
    for k in range(1, L+1):  # scalogram scales (1 to L in Python)
        for i in range(k+1, N-k):  # adjusted for 0-based indexing
            if x[i] > x[i-k] and x[i] > x[i+k]:
                m_max[k-1, i] = True
            if x[i] < x[i-k] and x[i] < x[i+k]:
                m_min[k-1, i] = True

    # Step 2: find the scale with the most local maxima (or local minima)
    # - row-wise summation (i.e. sum each row)
    gamma_max = np.sum(m_max, axis=1)  # axis=1 makes it row-wise
    gamma_min = np.sum(m_min, axis=1)  # axis=1 makes it row-wise

    # - find scale with the most local maxima (or local minima)
    lambda_max = np.argmax(gamma_max)
    lambda_min = np.argmax(gamma_min)

    # Step 3: Use lambda to remove all elements of m for which k>lambda
    m_max = m_max[:lambda_max+1, :]  # +1 because of 0-based indexing
    m_min = m_min[:lambda_min+1, :]  # +1 because of 0-based indexing

    # Step 4: Find peaks
    # - column-wise summation
    m_max_sum = np.sum(~m_max, axis=0)  # axis=0 makes it column-wise
    m_min_sum = np.sum(~m_min, axis=0)  # axis=0 makes it column-wise

    p = np.where(m_max_sum == 0)[0]  # find indices where sum is 0
    t = np.where(m_min_sum == 0)[0]  # find indices where sum is 0

    return p, t

In [ ]:
df_expanded['Peak_Index'] = df_expanded['PPG_Segment_Reflected'].apply(lambda x: msptd_beat_detector(x, fs=100))
print (df_expanded)

     Segment_Number  Age File_name  Glucose  \
0                 0   42   05a.csv      311   
1                 1   42   05a.csv      311   
2                 2   42   05a.csv      311   
3                 3   42   05a.csv      311   
4                 4   42   05a.csv      311   
..              ...  ...       ...      ...   
106               1   72   35b.csv      313   
107               2   72   35b.csv      313   
108               3   72   35b.csv      313   
109               4   72   35b.csv      313   
110               5   72   35b.csv      313   

                                           PPG_Segment  \
0    [13.356884608444958, -9.148327623185136, -28.7...   
1    [39.13716395126545, 39.99351715385333, 41.2936...   
2    [43.935461835001455, 37.216446019551505, 30.22...   
3    [14.179323261683749, -2.3363617837503288, -22....   
4    [21.520734170059207, 30.81025673404984, 39.525...   
..                                                 ...   
106  [-1.4868188628763694, -1

In [ ]:
X = np.stack(df_train['Cropped_1s_Normalized'].values)
X_1 = np.stack(df_train['VPG_Signal_Normalized'].values)
X_2 = np.stack(df_train['APG_Signal_Normalized'].values)
y = df_train['Result'].values
y_1 = df_train['Case'].values
y_2 = df_train['dt'].values
y_3 = df_train['Segment_Number'].values
y_4 = df_train['Peak_Index'].values

X_train, X_temp, X_1_train, X_1_temp, X_2_train, X_2_temp, y_train, y_temp, y_1_train, y_1_temp, y_2_train, y_2_temp, y_3_train, y_3_temp, y_4_train, y_4_temp = train_test_split(X, X_1, X_2, y, y_1, y_2, y_3, y_4, test_size=0.3, random_state=42)
X_val, X_test, X_1_val, X_1_test, X_2_val, X_2_test, y_val, y_test, y_1_val, y_1_test, y_2_val, y_2_test, y_3_val, y_3_test, y_4_val, y_4_test = train_test_split(X_temp, X_1_temp, X_2_temp, y_temp, y_1_temp, y_2_temp, y_3_temp, y_4_temp, test_size=0.5, random_state=42)

print(f"Train set: {X_train.shape, y_train.shape}; Valid set: {X_val.shape, y_val.shape}; Test set: {X_test.shape, y_test.shape}")

Train set: ((116569, 100), (116569,)); Valid set: ((24979, 100), (24979,)); Test set: ((24980, 100), (24980,))


In [ ]:
def Conv_1D_Block(x, filters, kernel_size=3, strides=1):
    """ 1D Convolutional Block with BatchNormalization and ReLU """
    x = tf.keras.layers.Conv1D(filters, kernel_size, strides=strides, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return x

def Residual_Block(x, filters, downsample=False):
    """ Residual Block với Fix lỗi shape mismatch """

    # 🔹 Shortcut Connection (1x1 Conv nếu cần downsample hoặc đổi filters)
    shortcut = x
    if downsample or x.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv1D(filters, 1, strides=2 if downsample else 1, padding="same", kernel_initializer="he_normal")(x)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    # 🔹 Conv1D → BatchNorm → ReLU
    x = tf.keras.layers.Conv1D(filters, kernel_size=3, strides=2 if downsample else 1, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    # 🔹 Conv1D (Không BatchNorm & ReLU sau đó)
    x = tf.keras.layers.Conv1D(filters, kernel_size=3, padding="same", kernel_initializer="he_normal")(x)
    # x = tf.keras.layers.BatchNormalization()(x)

    # 🔹 Skip Connection + ReLU Activation
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('relu')(x)

    return x

class ResNet34_1D:
    def __init__(self, length, num_channel, num_filters=64, problem_type='Regression', output_nums=1, dropout_rate=0.5):
        self.length = length
        self.num_channel = num_channel
        self.num_filters = num_filters
        self.problem_type = problem_type
        self.output_nums = output_nums
        self.dropout_rate = dropout_rate

    def MLP(self, x):
        """ Fully Connected Layers """
        x = tf.keras.layers.Flatten(name='flatten')(x)

        # 🔹 Dense 256 + Dropout
        #x = tf.keras.layers.Dense(256, activation='relu', name='dense_256')(x)
        #x = tf.keras.layers.Dropout(self.dropout_rate, name='dropout_1')(x)

        # 🔹 Dense 128 + Dropout
        x = tf.keras.layers.Dense(128, activation='relu', name='dense_128')(x)
        x = tf.keras.layers.Dropout(self.dropout_rate, name='dropout_2')(x)

        # 🔹 Output Layer (Regression or Classification)
        activation = 'softmax' if self.problem_type == 'Classification' else 'linear'
        outputs = tf.keras.layers.Dense(self.output_nums, activation=activation, name="output")(x)

        return outputs

    def build_resnet34(self):
        inputs = tf.keras.Input((self.length, self.num_channel))

        # 🔹 Conv1D + BatchNorm trước khi vào residual block
        x = Conv_1D_Block(inputs, self.num_filters, kernel_size=3, strides=1)

        # 🔹 Residual Blocks theo paper
        x = Residual_Block(x, 4)
        x = Residual_Block(x, 8)
        x = Residual_Block(x, 16, downsample=True)
        x = Residual_Block(x, 32, downsample=True)
        x = Residual_Block(x, 64, downsample=True)
        # 🔹 Fully Connected Layers
        outputs = self.MLP(x)

        # 🔹 Build Model
        model = tf.keras.Model(inputs, outputs)
        return model


length = 100
num_channel = 1
dropout_rate = 0.1

In [ ]:
import tensorflow as tf

def Conv_1D_Block(x, filters, kernel_size=3, strides=1):
    x = tf.keras.layers.Conv1D(filters, kernel_size, strides=strides, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return x

def Residual_Block(x, filters, downsample=False):
    # 🔹 Shortcut Connection (1x1 Conv nếu cần downsample hoặc đổi filters)
    shortcut = x
    if downsample or x.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv1D(filters, 1, strides=2 if downsample else 1, padding="same", kernel_initializer="he_normal")(x)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    # 🔹 Conv1D → BatchNorm → ReLU
    x = tf.keras.layers.Conv1D(filters, kernel_size=3, strides=2 if downsample else 1, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    # 🔹 Conv1D (Không BatchNorm & ReLU sau đó)
    x = tf.keras.layers.Conv1D(filters, kernel_size=3, padding="same", kernel_initializer="he_normal")(x)
    # x = tf.keras.layers.BatchNormalization()(x)

    # 🔹 Skip Connection + ReLU Activation
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('relu')(x)

    return x

class ResNet34_1D:
    def __init__(self, length, num_channel, num_filters=64, problem_type='Regression', output_nums=1, dropout_rate=0.5):
        self.length = length
        self.num_channel = num_channel
        self.num_filters = num_filters
        self.problem_type = problem_type
        self.output_nums = output_nums
        self.dropout_rate = dropout_rate

    def MLP(self, x):
        """ Fully Connected Layers """
        x = tf.keras.layers.Flatten(name='flatten')(x)

        # 🔹 Dense 256 + Dropout
        x = tf.keras.layers.Dense(256, activation='relu', name='dense_256')(x)
        x = tf.keras.layers.Dropout(self.dropout_rate, name='dropout_1')(x)

        # 🔹 Dense 128 + Dropout
        x = tf.keras.layers.Dense(128, activation='relu', name='dense_128')(x)
        x = tf.keras.layers.Dropout(self.dropout_rate, name='dropout_2')(x)

        activation = 'softmax' if self.problem_type == 'Classification' else 'linear'
        outputs = tf.keras.layers.Dense(self.output_nums, activation=activation, name="output")(x)

        return outputs

    def build_resnet34(self):
        inputs = tf.keras.Input((self.length, self.num_channel))

        x = Conv_1D_Block(inputs, self.num_filters, kernel_size=3, strides=1)

        x = Residual_Block(x, 64)
        x = Residual_Block(x, 64)
        x = Residual_Block(x, 64)

        x = Residual_Block(x, 128, downsample=True)
        x = Residual_Block(x, 128)
        x = Residual_Block(x, 128)
        x = Residual_Block(x, 128)

        x = Residual_Block(x, 256, downsample=True)
        x = Residual_Block(x, 256)
        x = Residual_Block(x, 256)
        x = Residual_Block(x, 256)
        x = Residual_Block(x, 256)
        x = Residual_Block(x, 256)

        x = Residual_Block(x, 512, downsample=True)
        x = Residual_Block(x, 512)
        x = Residual_Block(x, 512)

        outputs = self.MLP(x)

        model = tf.keras.Model(inputs, outputs)
        return model
length = 100 # Dữ liệu PPG 1D
num_channel = 1
dropout_rate = 0.3

In [ ]:
import tensorflow as tf
import itertools

param_grid_1 = {
    'num_layers':  [[3, 4, 6, 3]],
    'num_filters': [[8, 16, 32, 64]],
    'final_layer_size': [[256, 128]],
    'learning_rate': [1e-2],
    'optimizer' : ['sgd'],
    'dropout' : [0.1]
}

def generate_valid_configs(param_grid):
    """
    Generate valid configurations where num_layers and num_filters lengths match.
    """
    configs = [
        (nl, nf, fls, lr, opt, do)
        for nl, nf, fls, lr, opt, do in itertools.product(
            param_grid['num_layers'],
            param_grid['num_filters'],
            param_grid['final_layer_size'],
            param_grid['learning_rate'],
            param_grid['optimizer'],
            param_grid['dropout']
        )
        if len(nl) == len(nf)
    ]
    return configs

# Example usage
all_configs_1 = generate_valid_configs(param_grid_1)


# Optionally, combine all grids into one list
all_configs = (
    all_configs_1
)

print(f"Total valid configurations: {len(all_configs)}")

def Conv_1D_Block(x, filters, kernel_size=3, strides=1):
    x = tf.keras.layers.Conv1D(filters, kernel_size, strides=strides, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return x

def Residual_Block(x, filters, downsample=False):
    # 🔹 Shortcut Connection (1x1 Conv nếu cần downsample hoặc đổi filters)
    shortcut = x
    if downsample or x.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv1D(filters, 1, strides=2 if downsample else 1, padding="same", kernel_initializer="he_normal")(x)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    # 🔹 Conv1D → BatchNorm → ReLU
    x = tf.keras.layers.Conv1D(filters, kernel_size=3, strides=2 if downsample else 1, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    # 🔹 Conv1D (Không BatchNorm & ReLU sau đó)
    x = tf.keras.layers.Conv1D(filters, kernel_size=3, padding="same", kernel_initializer="he_normal")(x)
    # x = tf.keras.layers.BatchNormalization()(x)

    # 🔹 Skip Connection + ReLU Activation
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('relu')(x)

    return x

class ResNet34_1D:
    def __init__(self, length, num_channel, num_layers, final_layer_size, num_filters_residual, num_filters=64, problem_type='Regression', output_nums=1, dropout_rate=0.5):
        self.length = length
        self.num_channel = num_channel
        self.num_filters = num_filters
        self.num_filters_residual = num_filters_residual
        self.problem_type = problem_type
        self.output_nums = output_nums
        self.dropout_rate = dropout_rate
        self.num_layers = num_layers
        self.final_layer_size = final_layer_size

    def MLP(self, x, size_list):
        """ Fully Connected Layers """
        x = tf.keras.layers.Flatten(name='flatten')(x)

        for i, size in enumerate(size_list):
        # 🔹 Dense 256 + Dropout
          x = tf.keras.layers.Dense(size, activation='relu', name=f'dense_{size}')(x)
          x = tf.keras.layers.Dropout(self.dropout_rate, name=f'dropout_{i}')(x)

        activation = 'softmax' if self.problem_type == 'Classification' else 'linear'
        outputs = tf.keras.layers.Dense(self.output_nums, activation=activation, name="output")(x)

        return outputs

    def build_resnet34(self):
        inputs = tf.keras.Input((self.length, self.num_channel))

        x = Conv_1D_Block(inputs, self.num_filters, kernel_size=3, strides=1)

        for stage_idx, (filters, layers) in enumerate(zip(self.num_filters_residual, self.num_layers)):
            for block_idx in range(layers):
              downsample = (stage_idx > 0 and block_idx == 0)
              x = Residual_Block(x, filters, downsample=downsample)
        outputs = self.MLP(x, self.final_layer_size)
        model = tf.keras.Model(inputs, outputs)
        return model

Total valid configurations: 1


In [ ]:
from tensorflow.keras.callbacks import Callback, ModelCheckpoint
import tensorflow as tf
import os
import random
import numpy as np
from tqdm.keras import TqdmCallback
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

'''def get_flops_tf2(model, input_shape=None):
    """
    Returns estimated FLOPs for `model` (int).
    input_shape: e.g. (length, channels) or model.input_shape[1:]
    Uses batch_size=1 for flops estimation.
    """
    # Determine input shape
    if input_shape is None:
        # Try model.input_shape; may be (None, ...) or list/tuple for multi-input
        try:
            ishape = model.input_shape
            if isinstance(ishape, list):  # multi-input
                # use first input
                ishape = ishape[0]
            input_shape = tuple(ishape[1:])  # drop batch dim
        except Exception:
            raise ValueError("input_shape not provided and model.input_shape unavailable.")

    # Make a concrete function
    try:
        func = tf.function(lambda x: model(x))
        concrete = func.get_concrete_function(tf.TensorSpec([1, *list(input_shape)], tf.float32))
    except Exception as e:
        # fallback for models that need additional args (rare)
        raise RuntimeError("Failed to get concrete function: " + str(e))

    # Freeze the graph
    frozen_func = convert_variables_to_constants_v2(concrete)
    graph_def = frozen_func.graph.as_graph_def()

    # Create a new graph and import the frozen graph_def
    tf.compat.v1.reset_default_graph()
    g = tf.Graph()
    with g.as_default():
        tf.import_graph_def(graph_def, name='')

        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        # Profile the graph
        try:
            flops_profile = tf.compat.v1.profiler.profile(
                g,
                run_meta=run_meta,
                cmd='op',
                options=opts
            )
        except Exception as e:
            # If profiler fails, raise with context
            raise RuntimeError("TF profiler failed: " + str(e))

        if flops_profile is None:
            return 0
        return int(flops_profile.total_float_ops)'''


def safe_name(lst):
    return "-".join(map(str, lst)) if isinstance(lst, list) else str(lst)

'''def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)'''

# Custom checkpoint to only save within certain epoch ranges
'''class RangeCheckpoint(Callback):
    def __init__(self, filepath, monitor='val_loss', mode='min', epoch_range=(1, 50)):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.epoch_start, self.epoch_end = epoch_range
        self.best = np.inf if mode == 'min' else -np.inf

    def on_epoch_end(self, epoch, logs=None):
        epoch_num = epoch + 1
        current = logs.get(self.monitor)
        if self.epoch_start <= epoch_num <= self.epoch_end:
            if (self.mode == 'min' and current < self.best) or (self.mode == 'max' and current > self.best):
                self.best = current
                self.model.save(self.filepath)
                print(f"\n✅ Best model updated (epoch {epoch_num}, {self.monitor}={current:.5f}) → {self.filepath}")'''
NUM_MODELS_PER_CONFIG = 13

# Training loop
for i, (num_layers, num_filters_residual, final_layer_size, lr, opt, do) in enumerate(all_configs, 1):

    for run in range(1, NUM_MODELS_PER_CONFIG + 1):

        print(f"\n🔹 Training config {i}/{len(all_configs)} — run {run}/{NUM_MODELS_PER_CONFIG}:")
        print(f"num_layers={num_layers}, num_filters={num_filters_residual}, final_layer_size={final_layer_size}")

        tf.keras.backend.clear_session()

        model = ResNet34_1D(
            length=98,
            num_channel=1,
            num_layers=num_layers,
            final_layer_size=final_layer_size,
            num_filters_residual=num_filters_residual,
            dropout_rate=do,
        ).build_resnet34()

        # Optimizer selection
        if opt == 'adam':
            optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
        elif opt == 'sgd':
            optimizer = tf.keras.optimizers.SGD(learning_rate=lr)
        elif opt == 'rmsprop':
            optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)

        model.compile(
            optimizer=optimizer,
            loss=tf.keras.losses.MeanAbsoluteError(),
            metrics=[tf.keras.metrics.MeanAbsoluteError()]
        )

        base_path = "/content/drive/MyDrive/customized_normalized_onsets_APG_models"
        model_prefix = (
            f"APG_{safe_name(num_layers)}_{safe_name(num_filters_residual)}_"
            f"{safe_name(final_layer_size)}_{safe_name(lr)}_{safe_name(opt)}_"
            f"{safe_name(do)}_run{run+7}"
        )

        callbacks = [
            ModelCheckpoint(
                filepath=os.path.join(base_path, f"{model_prefix}.keras"),
                monitor='val_loss',
                save_best_only=True,
                mode='min',
                verbose=0
            ),
            TqdmCallback(verbose=1)
        ]

        history = model.fit(
            X_2_train, y_train,
            epochs=100,
            batch_size=256,
            validation_data=(X_2_val, y_val),
            shuffle=True,
            verbose=0,
            callbacks=callbacks
        )

        print(f"✅ Completed config {i}, run {run}.\n")


🔹 Training config 1/1 — run 1/13:
num_layers=[3, 4, 6, 3], num_filters=[8, 16, 32, 64], final_layer_size=[256, 128]


0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

✅ Completed config 1, run 1.


🔹 Training config 1/1 — run 2/13:
num_layers=[3, 4, 6, 3], num_filters=[8, 16, 32, 64], final_layer_size=[256, 128]


0epoch [00:00, ?epoch/s]

...

0batch [00:00, ?batch/s]

✅ Completed config 1, run 12.


🔹 Training config 1/1 — run 13/13:
num_layers=[3, 4, 6, 3], num_filters=[8, 16, 32, 64], final_layer_size=[256, 128]


0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

✅ Completed config 1, run 13.



In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tqdm.notebook import tqdm
from tqdm.keras import TqdmCallback
import tensorflow as tf

for i in range(20):  # v1 to v20
    print(f"Starting training iteration {i+1}/20 from scratch")

    # Clear previous model from memory
    tf.keras.backend.clear_session()

    # Define the model from scratch
    model = ResNet34_1D(length, num_channel, dropout_rate=dropout_rate).build_resnet34()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
        loss=tf.keras.losses.MeanAbsoluteError(),
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )

    # Set adaptive model path
    model_path = f'/content/drive/MyDrive/APG_Onset_Original_v{i+1}.keras'

    # Define callbacks (tqdm for notebook)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, verbose=0, mode='min'),
        ModelCheckpoint(model_path, verbose=0, monitor='val_loss', save_best_only=True, mode='min'),
        TqdmCallback(verbose=1)  # uses notebook-style progress bar in Colab
    ]

    # Train the model from scratch
    history = model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=256,
        verbose=0,  # disable default Keras logs
        validation_data=(X_val, y_val),
        shuffle=True,
        callbacks=callbacks
    )

    print(f"Training iteration {i+1} completed. Model saved as {model_path}\n")


Starting training iteration 1/20 from scratch


0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

Training iteration 1 completed. Model saved as /content/drive/MyDrive/APG_Onset_Original_v1.keras

Starting training iteration 2/20 from scratch


0epoch [00:00, ?epoch/s]

...

0batch [00:00, ?batch/s]

Training iteration 19 completed. Model saved as /content/drive/MyDrive/APG_Onset_Original_v19.keras

Starting training iteration 20/20 from scratch


0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

Training iteration 20 completed. Model saved as /content/drive/MyDrive/APG_Onset_Original_v20.keras



In [ ]:
valid_csv_file = r'/content/drive/MyDrive/Data/New_MUST_2peaks_Onset.pkl'
df_valid = pd.read_pickle(valid_csv_file)
print (df_valid)

                                           Cropped_1s  Peak_Index Onset_Index  \
0   [0.8825383094588373, -0.6088356922443359, -1.9...         144   (82, 162)   
1   [5.368665641657043, 3.972900888167555, 2.59853...         295  (238, 314)   
2   [4.49780608174009, 3.134278005358381, 1.800455...         452  (391, 470)   
3   [7.403450286756618, 6.038496596580257, 4.52995...         603  (545, 621)   
4   [8.566463910289341, 7.056559256231465, 5.38597...         753  (695, 769)   
..                                                ...         ...         ...   
67  [-0.5057752303393482, 0.01959587751595928, 0.4...         270  (255, 291)   
68  [0.4805772360868463, 0.65809297376244, 0.75258...         361  (329, 381)   
69  [-2.6799098765655387, -2.8278353358700508, -2....         471  (455, 489)   
70  [-0.4849902198962859, -0.460205503120492, -0.4...         576  (561, 627)   
71  [0.6665445729077198, 0.47776148660520557, 0.24...         681  (627, 698)   

    Middle_Index  Glucose  

In [ ]:
valid_csv_file_2 = r'/content/drive/MyDrive/Data/New_Green_Inverted_Peak.pkl'
df_valid_2 = pd.read_pickle(valid_csv_file_2)
print (df_valid_2)

    File_name  Glucose  Segment_Number  Peak_Index  \
0     06a.csv       97               0         327   
1     06a.csv       97               0         428   
2     06a.csv       97               0         534   
3     06a.csv       97               0         642   
4     06a.csv       97               1          57   
..        ...      ...             ...         ...   
843   37a.csv      176               2         791   
844   37a.csv      176               3         851   
845   37b.csv      176               0          95   
846   37b.csv      176               2         437   
847   37b.csv      176               3         717   

                                            Cropped_1s  \
0    [-37.69894414727537, -40.851370683685765, -43....   
1    [-46.45901090751395, -48.6975446527504, -51.94...   
2    [-37.66747891673276, -37.81155816704377, -38.6...   
3    [-54.13160548874616, -54.671854080041456, -55....   
4    [-37.59212835007184, -41.103165122796035, -44....   
.. 

In [ ]:
valid_csv_file_3 = r'/content/drive/MyDrive/Data/Green_Inverted_Onset_v2.pkl'
df_valid_3 = pd.read_pickle(valid_csv_file_3)
print (df_valid_3)

                                            Cropped_1s  Peak_Index  \
0    [-3.002764414574935, -8.000947137837105, -11.7...         146   
1    [-43.9325485614622, -44.47639870997052, -43.44...         657   
2    [32.68479051832884, 24.69729298630475, 18.1779...         868   
3    [-150.17061004726563, -158.69337197959325, -16...         129   
4    [69.86364977782648, 83.62747769944863, 88.9382...         244   
..                                                 ...         ...   
256  [-18.642593122768208, -18.885189456384364, -20...         510   
257  [4.464590686648956, -2.4617851618173057, -9.34...         637   
258  [-41.10158132010425, -44.29591232620037, -48.5...         341   
259  [-27.939839696605656, -28.685685523912014, -31...         532   
260  [-31.265260102532213, -38.00939200503329, -46....         793   

    Onset_Index  Middle_Index  Segment_Number  Result  Age File_name  \
0    (133, 201)           167               1     311   42   05a.csv   
1    (614, 712)

In [ ]:
X_must = np.stack(df_valid['Cropped_1s_Normalized'].values)
X_1_must = np.stack(df_valid['VPG_Signal_Normalized'].values)
X_2_must = np.stack(df_valid['APG_Signal_Normalized'].values)
y_must = df_valid['Glucose'].values
y_1_must = df_valid['Index_Peak'].values
print (X_must.shape, y_must.shape)

(72, 100) (72,)


In [ ]:
X_Green = np.stack(df_valid_2['Cropped_1s_Normalized'].values)
X_1_Green = np.stack(df_valid_2['VPG_Signal_Normalized'].values)
X_2_Green = np.stack(df_valid_2['APG_Signal_Normalized'].values)
y_Green = df_valid_2['Glucose'].values
y_1_Green = df_valid_2['File_name'].values
y_2_Green = df_valid_2['Segment_Number'].values
y_3_Green = df_valid_2['Peak_Index'].values
print (X_Green.shape, y_Green.shape)

(848, 100) (848,)


In [ ]:
X_Green_2 = np.stack(df_valid_3['Cropped_1s_Normalized'].values)
X_1_Green_2 = np.stack(df_valid_3['VPG_Signal_Normalized'].values)
X_2_Green_2 = np.stack(df_valid_3['APG_Signal_Normalized'].values)
y_Green_2 = df_valid_3['Result'].values
y_1_Green_2 = df_valid_3['File_name'].values
y_2_Green_2 = df_valid_3['Segment_Number'].values
y_3_Green_2 = df_valid_3['Peak_Index'].values
print (X_Green_2.shape, y_Green_2.shape)

(261, 100) (261,)


In [ ]:
import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    "APG": {
        "path": "/content/drive/MyDrive/Models/Onsets_Normalized_Quantized/APG_weight_int8_io_float32.tflite",
        "data": {"test": X_3_test, "must": X_3_must, "Green": X_3_Green, "Green_2": X_3_Green_2, "Green_3": X_3_Green_3}
    },
    "PPG": {
        "path": "/content/drive/MyDrive/Models/Onsets_Normalized_Quantized/PPG_weight_int8_io_float32.tflite",
        "data": {"test": X_test, "must": X_must, "Green": X_Green, "Green_2": X_Green_2, "Green_3": X_Green_3}
    },
    "VPG": {
        "path": "/content/drive/MyDrive/Models/Onsets_Normalized_Quantized/VPG_weight_int8_io_float32.tflite",
        "data": {"test": X_2_test, "must": X_2_must, "Green": X_2_Green, "Green_2": X_2_Green_2, "Green_3": X_2_Green_3}
    }
}

y_data = {
    "test": y_test,
    "must": y_must,
    "Green": y_Green,
    "Green_2": y_Green_2,
    "Green_3": y_Green_3
}

def predict_model(model_path, X_data):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    preds = []
    for i in range(len(X_data)):
        sample = X_data[i].reshape(1, X_data.shape[1], 1).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        pred = interpreter.get_tensor(output_details[0]['index'])
        preds.append(pred.flatten()[0])
    return np.array(preds)

all_preds = {ds_name: [] for ds_name in models["APG"]["data"].keys()}

for model_name, info in models.items():
    for ds_name, X_data in info["data"].items():
        preds = predict_model(info["path"], X_data)
        all_preds[ds_name].append(preds)

averaged_preds = {}
for ds_name, preds_list in all_preds.items():
    # check lengths match
    if all(len(p) == len(preds_list[0]) for p in preds_list):
        averaged_preds[ds_name] = np.mean(preds_list, axis=0)
    else:
        print(f"Warning: Cannot average {ds_name}, sample lengths differ")

results_all = []
for name, predictions in averaged_preds.items():
  y = y_data[name]
  def clarke_error_grid_zone(y_true, y_pred):
    if y_true < 70 and y_pred < 70:
        return "A"
    elif 0.8 * y_true <= y_pred <= 1.2 * y_true:
        return "A"
    elif (y_true >= 70 and y_true <= 290 and y_pred > y_true + 110) or \
        (y_true >= 130 and y_true <= 180 and y_pred < (7/5) * y_true - 182):
        return "C"
    elif (y_true > 240 and 70 <= y_pred <= 180) or \
        (y_true < 58 and 70 <= y_pred <= 180) or \
        (y_true >= 58 and y_true <= 70 and y_pred > 1.2 * y_true):
        return "D"
    elif (y_true > 180 and y_pred < 70) or (y_true < 70 and y_pred > 180):
        return "E"
    else:
        return "B"
  zones_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y, predictions)]
  zone_counts_test = pd.Series(zones_test).value_counts(normalize=True) * 100

  mse = mean_squared_error(y, predictions)
  rmse = np.sqrt(mse)
  mae = mean_absolute_error(y, predictions)
  mard = np.mean(np.abs((y - predictions) / y)) * 100
  r2 = r2_score(y, predictions)

  results_all.append({
      "Dataset": name,
      "MSE": mse,
      "RMSE": rmse,
      "MAE": mae,
      "MARD": mard,
      "R2": r2,
      "Zone_A": zone_counts_test.get("A", 0),
      "Zone_B": zone_counts_test.get("B", 0),
      "Zone_C": zone_counts_test.get("C", 0),
      "Zone_D": zone_counts_test.get("D", 0),
      "Zone_E": zone_counts_test.get("E", 0),

  })

results_df = pd.DataFrame(results_all)
results_csv_path = "/content/clarke_error_grid_evaluation_PPG_VPG_APG_Onset_Normalized_tflite.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"Evaluation results saved to {results_csv_path}")



/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluation results saved to /content/clarke_error_grid_evaluation_PPG_VPG_APG_Onset_Normalized_tflite.csv


In [ ]:
import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_path = "/content/drive/MyDrive/tmp_model_quantized/APG_Normalized/APG_3-4-6-3_8-16-32-64_64_0.01_adam_0.1_run2_weight_int8_io_float32.tflite"

def predict_model(model_path, X_data):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    preds = []
    for i in range(len(X_data)):
        sample = X_data[i].reshape(1, X_data.shape[1], 1).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        pred = interpreter.get_tensor(output_details[0]['index'])
        preds.append(pred.flatten()[0])
    return np.array(preds)

predictions = predict_model(model_path, X_Green)
predictions_2 = predict_model(model_path, X_Green_2)

def clarke_error_grid_zone(y_true, y_pred):
  if y_true < 70 and y_pred < 70:
      return "A"
  elif 0.8 * y_true <= y_pred <= 1.2 * y_true:
      return "A"
  elif (y_true >= 70 and y_true <= 290 and y_pred > y_true + 110) or \
      (y_true >= 130 and y_true <= 180 and y_pred < (7/5) * y_true - 182):
      return "C"
  elif (y_true > 240 and 70 <= y_pred <= 180) or \
      (y_true < 58 and 70 <= y_pred <= 180) or \
      (y_true >= 58 and y_true <= 70 and y_pred > 1.2 * y_true):
      return "D"
  elif (y_true > 180 and y_pred < 70) or (y_true < 70 and y_pred > 180):
      return "E"
  else:
      return "B"
zones_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_Green, predictions)]
zone_counts_test = pd.Series(zones_test).value_counts(normalize=True) * 100
zones_test_2 = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_Green_2, predictions_2)]
zone_counts_test_2 = pd.Series(zones_test_2).value_counts(normalize=True) * 100



/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.metrics import mean_squared_error

'''models_folder = "/content/drive/MyDrive/quant_tests"   # folder with many .tflite files
results_csv_path = "/content/drive/MyDrive/middle_3_PPG_tflite_val.csv"'''
full_path = '/content/drive/MyDrive/customized_normalized_onsets_PPG_models/best_PPG_onset_customized_normalized_quantized.tflite'
full_path_2 = '/content/drive/MyDrive/customized_normalized_onsets_VPG_models/best_VPG_onset_customized_normalized_quantized.tflite'
full_path_3 = '/content/drive/MyDrive/customized_normalized_onsets_APG_models/best_APG_onset_customized_normalized_quantized.tflite'
results_csv_path = "/content/drive/MyDrive/MUST_onsets_quantized.csv"

def run_tflite_inference(model_path, X):
    interpreter = tf.lite.Interpreter(
        model_path=model_path,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Auto-detect input shape
    input_shape = input_details[0]['shape']   # e.g. (1, 100, 1) or (1, 99, 1)

    preds = []
    for i in range(len(X)):
        sample = X[i].reshape(input_shape).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        prediction = interpreter.get_tensor(output_details[0]['index'])
        preds.append(prediction.flatten()[0])

    return np.array(preds)

def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mard = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mard

def clarke_error_grid_zone(y_true, y_pred):
    if y_true < 70 and y_pred < 70:
        return "A"
    elif 0.8 * y_true <= y_pred <= 1.2 * y_true:
        return "A"
    elif (y_true >= 70 and y_true <= 290 and y_pred > y_true + 110) or \
         (y_true >= 130 and y_true <= 180 and y_pred < (7/5) * y_true - 182):
        return "C"
    elif (y_true > 240 and 70 <= y_pred <= 180) or \
         (y_true < 58 and 70 <= y_pred <= 180) or \
         (y_true >= 58 and y_true <= 70 and y_pred > 1.2 * y_true):
        return "D"
    elif (y_true > 180 and y_pred < 70) or (y_true < 70 and y_pred > 180):
        return "E"
    else:
        return "B"

results_all = []

'''for root, dirs, files in os.walk(models_folder):
    for file in files:
        if file.endswith(".tflite"):
            full_path = os.path.join(root, file)
            print("🔍 Testing:", file)'''

'''file_size_bytes = os.path.getsize(full_path)
file_size_mb = file_size_bytes / (1024 * 1024)'''

'''filename_no_ext = os.path.splitext(file)[0]
parts = filename_no_ext.split("_")'''


predictions_P = run_tflite_inference(full_path, X_must)
predictions_V = run_tflite_inference(full_path_2, X_1_must)
predictions_A = run_tflite_inference(full_path_3, X_2_must)

zones_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_must, predictions_P)]

zones_1_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_must, predictions_V)]

zones_2_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_must, predictions_A)]


'''predictions_PVA = (run_tflite_inference(full_path, X_test) + run_tflite_inference(full_path_3, X_2_test) + run_tflite_inference(full_path_2, X_1_test))/3
predictions_PV = (run_tflite_inference(full_path, X_test) + run_tflite_inference(full_path_2, X_1_test))/2
predictions_PA = (run_tflite_inference(full_path, X_test) + run_tflite_inference(full_path_3, X_2_test))/2
predictions_VA = (run_tflite_inference(full_path_3, X_2_test) + run_tflite_inference(full_path_2, X_1_test))/2

predictions_P_must = run_tflite_inference(full_path, X_must)
predictions_V_must = run_tflite_inference(full_path_2, X_1_must)
predictions_A_must = run_tflite_inference(full_path_3, X_2_must)
predictions_PVA_must = (run_tflite_inference(full_path, X_must) + run_tflite_inference(full_path_3, X_2_must) +  run_tflite_inference(full_path_2, X_1_must))/3
predictions_PV_must = (run_tflite_inference(full_path, X_must) + run_tflite_inference(full_path_2, X_1_must))/2
predictions_PA_must = (run_tflite_inference(full_path, X_must) + run_tflite_inference(full_path_3, X_2_must))/2
predictions_VA_must = (run_tflite_inference(full_path_3, X_2_must) + run_tflite_inference(full_path_2, X_1_must))/2

predictions_P_green = run_tflite_inference(full_path, X_Green)
predictions_V_green = run_tflite_inference(full_path_2, X_1_Green)
predictions_A_green = run_tflite_inference(full_path_3, X_2_Green)
predictions_PVA_green = (run_tflite_inference(full_path, X_Green) + run_tflite_inference(full_path_3, X_2_Green) +  run_tflite_inference(full_path_2, X_1_Green))/3
predictions_PV_green = (run_tflite_inference(full_path, X_Green) + run_tflite_inference(full_path_2, X_1_Green))/2
predictions_PA_green = (run_tflite_inference(full_path, X_Green) + run_tflite_inference(full_path_3, X_2_Green))/2
predictions_VA_Green = (run_tflite_inference(full_path_3, X_2_Green) + run_tflite_inference(full_path_2, X_1_Green))/2

# Compute RMSE + MARD
rmse_P, mard_P = compute_metrics(y_test, predictions_P)
rmse_V, mard_V = compute_metrics(y_test, predictions_V)
rmse_A, mard_A = compute_metrics(y_test, predictions_A)
rmse_PV, mard_PV = compute_metrics(y_test, predictions_PV)
rmse_PVA, mard_PVA = compute_metrics(y_test, predictions_PVA)
rmse_PA, mard_PA = compute_metrics(y_test, predictions_PA)
rmse_VA, mard_VA = compute_metrics(y_test, predictions_VA)

rmse_P_must, mard_P_must = compute_metrics(y_must, predictions_P_must)
rmse_V_must, mard_V_must = compute_metrics(y_must, predictions_V_must)
rmse_A_must, mard_A_must = compute_metrics(y_must, predictions_A_must)
rmse_PV_must, mard_PV_must = compute_metrics(y_must, predictions_PV_must)
rmse_PVA_must, mard_PVA_must = compute_metrics(y_must, predictions_PVA_must)
rmse_PA_must, mard_PA_must = compute_metrics(y_must, predictions_PA_must)
rmse_VA_must, mard_VA_must = compute_metrics(y_must, predictions_VA_must)

rmse_P_green, mard_P_green = compute_metrics(y_Green, predictions_P_green)
rmse_V_green, mard_V_green = compute_metrics(y_Green, predictions_V_green)
rmse_A_green, mard_A_green = compute_metrics(y_Green, predictions_A_green)
rmse_PV_green, mard_PV_green = compute_metrics(y_Green, predictions_PV_green)
rmse_PVA_green, mard_PVA_green = compute_metrics(y_Green, predictions_PVA_green)
rmse_PA_green, mard_PA_green = compute_metrics(y_Green, predictions_PA_green)
rmse_VA_green, mard_VA_green = compute_metrics(y_Green, predictions_VA_green)'''


results_df = pd.DataFrame({
    "Case": y_1_test,
    "dt": y_2_test,
    "Segment_Number": y_3_test,
    "Peak_Index": y_4_test,
    "True_Glucose": y_test,
    "Prediction_PPG": predictions_P,
    "Zone_PPG": zones_test,
    "Prediction_VPG": predictions_V,
    "Zone_VPG": zones_1_test,
    "Prediction_APG": predictions_A,
    "Zone_APG": zones_2_test

})

'''results_all.append({
    "RMSE_PPG": rmse_P,
    "MARD_PPG": mard_P,
    "RMSE_PPG_MUST": rmse_P_must,
    "MARD_PPG_MUST": mard_P_must,
    "RMSE_PPG_GREEN": rmse_P_green,
    "MARD_PPG_GREEN": mard_P_green,
    "RMSE_VPG": rmse_V,
    "MARD_VPG": mard_V,
    "RMSE_VPG_MUST": rmse_V_must,
    "MARD_VPG_MUST": mard_V_must,
    "RMSE_VPG_GREEN": rmse_V_green,
    "MARD_VPG_GREEN": mard_V_green,
    "RMSE_APG": rmse_A,
    "MARD_APG": mard_A,
    "RMSE_APG_MUST": rmse_A_must,
    "MARD_APG_MUST": mard_A_must,
    "RMSE_APG_GREEN": rmse_A_green,
    "MARD_APG_GREEN": mard_A_green,
    "RMSE_PV": rmse_PV,
    "MARD_PV": mard_PV,
    "RMSE_PV_MUST": rmse_PV_must,
    "MARD_PV_MUST": mard_PV_must,
    "RMSE_PV_GREEN": rmse_PV_green,
    "MARD_PV_GREEN": mard_PV_green,
    "RMSE_PA": rmse_PA,
    "MARD_PA": mard_PA,
    "RMSE_PA_MUST": rmse_PA_must,
    "MARD_PA_MUST": mard_PA_must,
    "RMSE_PA_GREEN": rmse_PA_green,
    "MARD_PA_GREEN": mard_PA_green,
    "RMSE_VA": rmse_VA,
    "MARD_VA": mard_VA,
    "RMSE_VA_MUST": rmse_VA_must,
    "MARD_VA_MUST": mard_VA_must,
    "RMSE_VA_GREEN": rmse_VA_green,
    "MARD_VA_GREEN": mard_VA_green,
    "RMSE_PVA": rmse_PVA,
    "MARD_PVA": mard_PVA,
    "RMSE_PVA_MUST": rmse_PVA_must,
    "MARD_PVA_MUST": mard_PVA_must,
    "RMSE_PVA_GREEN": rmse_PVA_green,
    "MARD_PVA_GREEN": mard_PVA_green
})
results_df = pd.DataFrame(results_all)'''

results_df.to_csv(results_csv_path, index=False)

print("✅ Done! Results saved to:", results_csv_path)


✅ Done! Results saved to: /content/drive/MyDrive/MUST_onsets_quantized.csv


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.metrics import mean_squared_error

models_folder = "/content/drive/MyDrive/best_PPG_customized_non_normalized_tflite"   # folder with many .tflite files
results_csv_path = "/content/drive/MyDrive/best_PPG_customized_non_normalized_tflite.csv"

def run_tflite_inference(model_path, X):
    interpreter = tf.lite.Interpreter(
        model_path=model_path,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    )
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_shape = input_details[0]['shape']   # e.g. (1, 100, 1) or (1, 99, 1)

    preds = []
    for i in range(len(X)):
        sample = X[i].reshape(input_shape).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        prediction = interpreter.get_tensor(output_details[0]['index'])
        preds.append(prediction.flatten()[0])

    return np.array(preds)

def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mard = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mard

results_all = []
for root, dirs, files in os.walk(models_folder):
    for file in files:
        if file.endswith(".tflite"):
            full_path = os.path.join(root, file)
            print("🔍 Testing:", file)

            file_size_bytes = os.path.getsize(full_path)
            file_size_mb = file_size_bytes / (1024 * 1024)

            filename_no_ext = os.path.splitext(file)[0]
            parts = filename_no_ext.split("_")

            predictions_val = run_tflite_inference(full_path, X_val)
            predictions = run_tflite_inference(full_path, X_test)
            predictions_must = run_tflite_inference(full_path, X_must)
            predictions_green_tflite = run_tflite_inference(full_path, X_Green)

            rmse_val, mard_val = compute_metrics(y_val, predictions_val)
            rmse, mard = compute_metrics(y_test, predictions)
            rmse_must, mard_must = compute_metrics(y_must, predictions_must)
            rmse_green, mard_green = compute_metrics(y_Green, predictions_green_tflite)

            results_all.append({
                "Validation_RMSE": rmse_val,
                "Validation_MARD": mard_val,
                "RMSE": rmse,
                "MARD": mard,
                "RMSE_MUST": rmse_must,
                "MARD_MUST": mard_must,
                "RMSE_GREEN": rmse_green,
                "MARD_GREEN": mard_green,
                "Model_Size": file_size_mb,
                "Model_Name": parts[5]
            })

results_df = pd.DataFrame(results_all)
results_df.to_csv(results_csv_path, index=False)

print("✅ Done! Results saved to:", results_csv_path)


🔍 Testing: best_PPG_customized_non_quantized_rep3000.tflite
🔍 Testing: best_PPG_customized_non_quantized_rep2000.tflite


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


🔍 Testing: best_PPG_customized_non_quantized_rep4000.tflite


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


🔍 Testing: best_PPG_customized_non_quantized_rep5000.tflite


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Done! Results saved to: /content/drive/MyDrive/best_PPG_customized_non_normalized_tflite.csv


In [ ]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.metrics import mean_squared_error

models_folder = "/content/drive/MyDrive/customized_normalized_onsets_APG_models"   # folder with many .tflite files
results_csv_path = "/content/drive/MyDrive/APG_Onset_tflite.csv"

def run_tflite_inference(model_path, X):
    interpreter = tf.lite.Interpreter(model_path=model_path, experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    preds = []
    for i in range(len(X)):
        sample = X[i].reshape(1, 98, 1).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        prediction = interpreter.get_tensor(output_details[0]['index'])
        preds.append(prediction.flatten()[0])

    return np.array(preds)

def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mard = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mard

results_all = []

for root, dirs, files in os.walk(models_folder):
    for file in files:
        if file.endswith(".tflite"):
            full_path = os.path.join(root, file)
            print("🔍 Testing:", file)

            try:
              file_size_bytes = os.path.getsize(full_path)
            except FileNotFoundError:
              print("⚠️ Skipped missing file:", full_path)
              continue
            file_size_mb = file_size_bytes / (1024 * 1024)

            filename_no_ext = os.path.splitext(file)[0]
            parts = filename_no_ext.split("_")

            # Run inference
            predictions = run_tflite_inference(full_path, X_2_val)

            # Compute RMSE + MARD
            rmse, mard = compute_metrics(y_val, predictions)

            results_all.append({
                "Calibration dataset size": parts[7],
                "RMSE": rmse,
                "MARD": mard,
                "File_Size_MB": file_size_mb
            })

results_df = pd.DataFrame(results_all)
results_df.to_csv(results_csv_path, index=False)

print("✅ Done! Results saved to:", results_csv_path)


🔍 Testing: best_APG_onset_customized_normalized_non_quantized_rep2000.tflite


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


🔍 Testing: best_APG_onset_customized_normalized_non_quantized_rep3000.tflite


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


🔍 Testing: best_APG_onset_customized_normalized_non_quantized_rep4000.tflite


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


🔍 Testing: best_APG_onset_customized_normalized_non_quantized_rep5000.tflite


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Done! Results saved to: /content/drive/MyDrive/APG_Onset_tflite.csv


In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

'''models_dir = "/content/drive/MyDrive/VPG_ResNet_34_tmp_model"
output_path = "/content/drive/MyDrive/VPG_ResNet34_all.csv"'''
model_path = '/content/drive/MyDrive/customized_normalized_onsets_PPG_models/best_PPG_onset_customized_normalized_non_quantized.keras'
model_path_1 = '/content/drive/MyDrive/customized_normalized_onsets_VPG_models/best_VPG_onset_customized_normalized_non_quantized.keras'
model_path_2 = '/content/drive/MyDrive/customized_normalized_onsets_APG_models/best_APG_onset_customized_normalized_non_quantized.keras'
output_path = "/content/drive/MyDrive/Predictions/Onsets/Non-quantized/MUST_onsets_non_quantized.csv"

def clarke_error_grid_zone(y_true, y_pred):
    if y_true < 70 and y_pred < 70:
        return "A"
    elif 0.8 * y_true <= y_pred <= 1.2 * y_true:
        return "A"
    elif (y_true >= 70 and y_true <= 290 and y_pred > y_true + 110) or \
         (y_true >= 130 and y_true <= 180 and y_pred < (7/5) * y_true - 182):
        return "C"
    elif (y_true > 240 and 70 <= y_pred <= 180) or \
         (y_true < 58 and 70 <= y_pred <= 180) or \
         (y_true >= 58 and y_true <= 70 and y_pred > 1.2 * y_true):
        return "D"
    elif (y_true > 180 and y_pred < 70) or (y_true < 70 and y_pred > 180):
        return "E"
    else:
        return "B"

results_all = []

'''for root, dirs, files in os.walk(models_dir):
  for model_file in files:
        if not model_file.endswith(".keras"):
            continue

        model_path = os.path.join(root, model_file)

        model_name = os.path.splitext(model_file)[0]  # name without extension

        print(f"Processing model: {model_name}")'''


'''file_size_bytes = os.path.getsize(model_path)
file_size_mb = file_size_bytes / (1024 * 1024)

parts = model_name.split("_")'''

model = load_model(model_path)
model_1 = load_model(model_path_1)
model_2 = load_model(model_path_2)

predictions = model.predict(X_must).flatten()
predictions_1 = model_1.predict(X_1_must).flatten()
predictions_2 = model_2.predict(X_2_must).flatten()

zones_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_must, predictions)]

zones_1_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_must, predictions_1)]

zones_2_test = [clarke_error_grid_zone(y_t, y_p) for y_t, y_p in zip(y_must, predictions_2)]


'''predictions_PV = (predictions + predictions_1)/2
predictions_PA = (predictions + predictions_2)/2
predictions_VA = (predictions_2 + predictions_1)/2
predictions_PVA = (predictions + predictions_2 + predictions_1)/3

predictions_must = model.predict(X_must).flatten()
predictions_1_must = model_1.predict(X_1_must).flatten()
predictions_2_must = model_2.predict(X_2_must).flatten()
predictions_PV_must = (predictions_must + predictions_1_must)/2
predictions_PA_must = (predictions_must + predictions_2_must)/2
predictions_VA_must = (predictions_2_must + predictions_1_must)/2
predictions_PVA_must = (predictions_must + predictions_2_must + predictions_1_must)/3

predictions_green = model.predict(X_Green).flatten()
predictions_1_green = model_1.predict(X_1_Green).flatten()
predictions_2_green = model_2.predict(X_2_Green).flatten()
predictions_PV_green = (predictions_green + predictions_1_green)/2
predictions_PA_green = (predictions_green + predictions_2_green)/2
predictions_VA_green = (predictions_2_green + predictions_1_green)/2
predictions_PVA_green = (predictions_green + predictions_2_green + predictions_1_green)/3


# Regression metrics helper
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mard = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mse, rmse, mard

mse_test_PV, rmse_test_PV, mard_test_PV= metrics(y_test, predictions_PV)
mse_test_PA, rmse_test_PA, mard_test_PA= metrics(y_test, predictions_PA)
mse_test_VA, rmse_test_VA, mard_test_VA= metrics(y_test, predictions_VA)
mse_test_PVA, rmse_test_PVA, mard_test_PVA= metrics(y_test, predictions_PVA)

mse_must_PV, rmse_must_PV, mard_must_PV = metrics(y_must, predictions_PV_must)
mse_must_PA, rmse_must_PA, mard_must_PA = metrics(y_must, predictions_PA_must)
mse_must_VA, rmse_must_VA, mard_must_VA = metrics(y_must, predictions_VA_must)
mse_must_PVA, rmse_must_PVA, mard_must_PVA= metrics(y_must, predictions_PVA_must)

mse_green_PV, rmse_green_PV, mard_green_PV = metrics(y_Green, predictions_PV_green)
mse_green_PA, rmse_green_PA, mard_green_PA = metrics(y_Green, predictions_PA_green)
mse_green_VA, rmse_green_VA, mard_green_VA = metrics(y_Green, predictions_VA_green)
mse_green_PVA, rmse_green_PVA, mard_green_PVA = metrics(y_Green, predictions_PVA_green)'''
print (y_1_test.shape)

results_df = pd.DataFrame({
    "True_Glucose": y_must,
    "Prediction_PPG": predictions,
    "Zone_PPG": zones_test,
    "Prediction_VPG": predictions_1,
    "Zone_VPG": zones_1_test,
    "Prediction_APG": predictions_2,
    "Zone_APG": zones_2_test

})

'''results_all.append({
    "rmse_test_PV": rmse_test_PV,
    "mard_test_PV": mard_test_PV,
    "rmse_test_PA": rmse_test_PA,
    "mard_test_PA": mard_test_PA,
    "rmse_test_VA": rmse_test_VA,
    "mard_test_VA": mard_test_VA,
    "rmse_test_PVA": rmse_test_PVA,
    "mard_test_PVA": mard_test_PVA,
    "rmse_must_PV": rmse_must_PV,
    "mard_must_PV": mard_must_PV,
    "rmse_must_PA": rmse_must_PA,
    "mard_must_PA": mard_must_PA,
    "rmse_must_VA": rmse_must_VA,
    "mard_must_VA": mard_must_VA,
    "rmse_must_PVA": rmse_must_PVA,
    "mard_must_PVA": mard_must_PVA,
    "rmse_green_PV": rmse_green_PV,
    "mard_green_PV": mard_green_PV,
    "rmse_green_PA": rmse_green_PA,
    "mard_green_PA": mard_green_PA,
    "rmse_green_VA": rmse_green_VA,
    "mard_green_VA": mard_green_VA,
    "rmse_green_PVA": rmse_green_PVA,
    "mard_green_PVA": mard_green_PVA
})
results_all = pd.DataFrame(results_all)'''

results_df.to_csv(output_path, index=False)
print(f"\n✅ CSV saved to {output_path}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step


1/3 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step

3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step
(24980,)

✅ CSV saved to /content/drive/MyDrive/Predictions/Onsets/Non-quantized/MUST_onsets_non_quantized.csv


In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models_dir = "/content/drive/MyDrive/customized_normalized_onsets_APG_models"
output_path = "/content/drive/MyDrive/APG_Onset_Normalized_Customized.csv"

def clarke_error_grid_zone(y_true, y_pred):
    if y_true < 70 and y_pred < 70:
        return "A"
    elif 0.8 * y_true <= y_pred <= 1.2 * y_true:
        return "A"
    elif (y_true >= 70 and y_true <= 290 and y_pred > y_true + 110) or \
         (y_true >= 130 and y_true <= 180 and y_pred < (7/5) * y_true - 182):
        return "C"
    elif (y_true > 240 and 70 <= y_pred <= 180) or \
         (y_true < 58 and 70 <= y_pred <= 180) or \
         (y_true >= 58 and y_true <= 70 and y_pred > 1.2 * y_true):
        return "D"
    elif (y_true > 180 and y_pred < 70) or (y_true < 70 and y_pred > 180):
        return "E"
    else:
        return "B"

results_all = []

for root, dirs, files in os.walk(models_dir):
  for model_file in files:
        if not model_file.endswith(".keras"):
            continue

        model_path = os.path.join(root, model_file)

        model_name = os.path.splitext(model_file)[0]  # name without extension

        print(f"Processing model: {model_name}")

        file_size_bytes = os.path.getsize(model_path)
        file_size_mb = file_size_bytes / (1024 * 1024)

        parts = model_name.split("_")

        model = load_model(model_path)

        predictions = model.predict(X_2_test).flatten()

        predictions_must = model.predict(X_2_must).flatten()

        predictions_green = model.predict(X_2_Green).flatten()


        def metrics(y_true, y_pred):
            mse = mean_squared_error(y_true, y_pred)
            rmse = np.sqrt(mse)
            mard = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
            return mse, rmse, mard

        mse_test, rmse_test, mard_test= metrics(y_test, predictions)
        mse_must, rmse_must, mard_must = metrics(y_must, predictions_must)
        mse_green, rmse_green, mard_green = metrics(y_Green, predictions_green)

        results_all.append({
            "ver": parts[7],
            "size": file_size_mb,
            "RMSE": rmse_test,
            "MARD": mard_test,
            "RMSE_MUST": rmse_must,
            "MARD_MUST": mard_must,
            "RMSE_GREEN": rmse_green,
            "MARD_GREEN": mard_green
        })


results_df = pd.DataFrame(results_all)
results_df.to_csv(output_path, index=False)
print(f"\n✅ CSV saved to {output_path}")


Processing model: APG_3-4-6-3_8-16-32-64_256-128_0.01_sgd_0.1_run8
781/781 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 662ms/step
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step
Processing model: APG_3-4-6-3_8-16-32-64_256-128_0.01_sgd_0.1_run9
781/781 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 660ms/step
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step
Processing model: APG_3-4-6-3_8-16-32-64_256-128_0.01_sgd_0.1_run10
781/781 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step
Processing model: APG_3-4-6-3_8-16-32-64_256-128_0.01_sgd_0.1_run11
781/781 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 646ms/step
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step
Processing model: APG_3-4-6-3_8-16-32-64_256-128_0.01_sgd_0.1_run12
781/781 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 656ms/step
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step
Processing model: APG_3-4-6-3_8-16-32-64_256-128_0.01_